# Ablation Study: SAMV-FC Large (Single random seed)
### Empirical Ablation Experiments on Dev & Test Sets (Paper Table 3)


| Ablation | Detail |
|----------|--------|
| Full | proposed model| 
| A1 | w/o auxiliary conflict branch |
| A2 | w/o NLI adaptation |
| A3 | w/o pair-pooling | 
| A4 | w/o logic-consistency loss |
| A5 | w/o diff markers | 


## 1. Environment & Google Drive Setup

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
ON_COLAB = True

# 2. Install dependencies
!pip install -q transformers datasets scikit-learn matplotlib sentencepiece accelerate "kagglehub[pandas-datasets]"

# 3. Core libraries
import os, re, json, time, math, random, shutil
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Experimental Configuration & Ablation Definitions

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DATASET_NAME = 'viwikifc'  # 'vifactcheck' | 'viwikifc'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KAGGLE_DATASET = 'thaonngyn/data-fc'
DATASET_FILES = {
    'vifactcheck': {
        'train': 'train-vifactcheck.csv',
        'dev': 'dev-vifactcheck.csv',
        'test': 'test-vifactcheck.csv',
    },
    'viwikifc': {
        'train': 'train-factcheckwiki.csv',
        'dev': 'dev-factcheckwiki.csv',
        'test': 'test-factcheckwiki.csv',
    },
}

BASE_DIR = '/content/drive/MyDrive/NCKH' if os.path.exists('/content/drive/MyDrive') else os.path.abspath('./colab_working')
WORKING_DIR = os.path.join(BASE_DIR, 'ablation_runs')
ABLATION_SEED = 1
OUT_DIR = os.path.join(WORKING_DIR, f'ablation_{DATASET_NAME}_seed{ABLATION_SEED}')
os.makedirs(OUT_DIR, exist_ok=True)
KEEP_CHECKPOINTS = False

FULL = {
    'name': 'full',
    'label': f'SAMV-FC Large (Full, seed {ABLATION_SEED})',
    'use_nli': True,
    'use_diff_markers': True,
    'use_pair_pooling': True,
    'use_conflict_head': True,
    'use_gate': True,
    'refuted_weight': 1.20,
    'label_smoothing': 0.10,
    'lambda_conflict': 0.35,
    'lambda_consistency': 0.20,
}

ABLATIONS = [
    {
        **FULL,
        'name': 'wo_auxiliary_conflict_branch',
        'label': 'w/o auxiliary conflict branch (single-task verdict)',
        'use_conflict_head': False,
        'use_gate': False,
        'lambda_conflict': 0.0,
        'lambda_consistency': 0.0,
    },
    {**FULL, 'name': 'wo_nli', 'label': 'w/o NLI intermediate fine-tuning', 'use_nli': False},
    {**FULL, 'name': 'wo_pair_pooling', 'label': 'w/o pair-pooling', 'use_pair_pooling': False},
    {**FULL, 'name': 'wo_consistency', 'label': 'w/o consistency loss', 'lambda_consistency': 0.0},
    {**FULL, 'name': 'wo_diff_markers', 'label': 'w/o diff markers (<diff>...</diff>)', 'use_diff_markers': False},
    {**FULL, 'name': 'full', 'label': f'SAMV-FC Large (Full, seed {ABLATION_SEED})'},
    {**FULL, 'name': 'wo_class_weight', 'label': 'w/o class weighting (uniform 1.0)', 'refuted_weight': 1.0},
    {**FULL, 'name': 'wo_label_smoothing', 'label': 'w/o label smoothing (0.0)', 'label_smoothing': 0.0},
    {**FULL, 'name': 'wo_weight_smoothing', 'label': 'w/o class weight & label smoothing (standard CE)', 'refuted_weight': 1.0, 'label_smoothing': 0.0},
]

FULL_REFERENCE_NPZ, FULL_REFERENCE_CKPT = None, None
for cdir in [os.path.join(BASE_DIR, 'main_runs'), BASE_DIR, '/content/drive/MyDrive/NCKH/main_runs']:
    p = Path(cdir)
    if p.exists():
        for npz in p.glob(f'**/*{DATASET_NAME}*/**/ensemble_probs_seed{ABLATION_SEED}.npz'):
            if npz.exists():
                FULL_REFERENCE_NPZ = str(npz); break
        if FULL_REFERENCE_NPZ: break
        for pt in p.glob(f'**/*{DATASET_NAME}*/**/best_multitask_xlmr_large_seed{ABLATION_SEED}.pt'):
            if pt.exists():
                FULL_REFERENCE_CKPT = str(pt); break
        if FULL_REFERENCE_CKPT: break

DEFAULT_FULL_METRICS = {
    'vifactcheck': {
        'dev_macro_f1': 0.8935, 'test_macro_f1': 0.8923, 'test_accuracy': 0.8923, 'conflict_slot_acc': 0.7336,
        'per_class_f1': {'Supported': 0.889, 'Refuted': 0.890, 'NEI': 0.913}
    },
    'viwikifc': {
        'dev_macro_f1': 0.8763, 'test_macro_f1': 0.8738, 'test_accuracy': 0.8739, 'conflict_slot_acc': 0.7420,
        'per_class_f1': {'Supported': 0.889, 'Refuted': 0.860, 'NEI': 0.881}
    },
}
FULL_REFERENCE_METRICS = DEFAULT_FULL_METRICS.get(DATASET_NAME, DEFAULT_FULL_METRICS['vifactcheck'])

RUN_ONLY = ['wo_auxiliary_conflict_branch']
ALIASES = {
    'wo_slot_detection': 'wo_auxiliary_conflict_branch',
    'wo_gate': 'wo_auxiliary_conflict_branch',
    'wo_conflict_branch': 'wo_auxiliary_conflict_branch',
}

def resolve_active_ablations(run_selection, all_ablations):
    if run_selection is None:
        return all_ablations
    if isinstance(run_selection, str):
        target_name = ALIASES.get(run_selection.strip(), run_selection.strip())
        return [a for a in all_ablations if a['name'] == target_name]
    if isinstance(run_selection, (list, tuple, set)):
        target_names = set(ALIASES.get(str(x).strip(), str(x).strip()) for x in run_selection)
        return [a for a in all_ablations if a['name'] in target_names]
    return all_ablations

ACTIVE_ABLATIONS = resolve_active_ablations(RUN_ONLY, ABLATIONS)

# Hyperparameters
NLI_SOURCE = 'xnli'
NLI_TRAIN_SAMPLES = 50000
NLI_EPOCHS = 2
FORCE_RETRAIN_NLI = False
NLI_LR = 1e-5
NLI_MAX_LEN = 128
NLI_BATCH_SIZE = 16
NLI_GRAD_ACCUM = 2
NLI_SEED = 42
USE_DIFF_MARKERS = True
USE_PAIR_POOLING = True
LABEL_SMOOTHING = 0.1
R_DROP_ALPHA = 1.0

MODEL_KEY = 'xlm-roberta-large'
MAX_LEN = 256
BATCH_SIZE = 4 if R_DROP_ALPHA > 0 else 8
GRAD_ACCUM = 16 // BATCH_SIZE
MAX_GRAD_NORM = 1.0
LR_BACKBONE = 5e-6
LR_HEADS = 1e-4
HEAD_WARMUP_EPOCHS = 1
NUM_EPOCHS = 8
DROPOUT = 0.1

TRAIN_LIMIT = None
DEV_LIMIT = None
TEST_LIMIT = None

print(f'Device: {DEVICE} | Dataset: {DATASET_NAME} | Active Ablations: {[a["name"] for a in ACTIVE_ABLATIONS]}')


## 3. Data Loader & Conflict Slot Labeling

In [ ]:
LABEL2ID = {'Supported': 0, 'Refuted': 1, 'NEI': 2}
ID2LABEL = {0: 'Supported', 1: 'Refuted', 2: 'NEI'}

CONFLICT_MAP = {'NONE': 0, 'TEMPORAL': 1, 'NUMERIC': 2, 'ENTITY': 3, 'NEGATION': 4}
ID2CONFLICT = {v: k for k, v in CONFLICT_MAP.items()}

def extract_years(text: str) -> List[int]:
    return sorted(set(int(y) for y in re.findall(r'\b((?:16|17|18|19|20)\d{2})\b', str(text))))

def extract_numbers(text: str) -> List[float]:
    nums = []
    for raw in re.findall(r'(?<!\w)(\d+(?:[.,]\d+)*)(?!\w)', str(text)):
        try:
            n = raw.replace(' ', '').replace('.', '').replace(',', '.')
            val = float(n)
            if val < 1600 or val > 2100:
                nums.append(val)
        except ValueError:
            pass
    return sorted(set(nums))

def detect_conflict_type(claim: str, evidence: str, label_str: str) -> str:
    if label_str != 'Refuted':
        return 'NONE'
    
    c_years, e_years = extract_years(claim), extract_years(evidence)
    if c_years and e_years and set(c_years) != set(e_years):
        return 'TEMPORAL'
    
    c_nums, e_nums = extract_numbers(claim), extract_numbers(evidence)
    if c_nums and e_nums and set(c_nums) != set(e_nums):
        return 'NUMERIC'
    
    negations = {'không', 'chưa', 'chẳng', 'không_thể', 'chưa_từng', 'phủ_nhận'}
    if any(w in claim.lower() for w in negations) != any(w in evidence.lower() for w in negations):
        return 'NEGATION'
    
    return 'ENTITY'

@dataclass
class MultiTaskSample:
    id: str
    claim: str
    evidence: str
    label_verdict: int
    label_conflict: int
    conflict_name: str

def find_dataset_file(fname: str) -> str:
    for d in ['/content/drive/MyDrive/NCKH/data', '/content/drive/MyDrive/data', '/content/data', './data', './dataset_raw']:
        p = Path(d)
        if p.exists():
            matches = list(p.rglob(fname))
            if matches:
                return str(matches[0])
    
    import kagglehub
    local_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    matches = list(local_root.rglob(fname))
    if matches:
        return str(matches[0])
    raise FileNotFoundError(f"File '{fname}' not found in Drive or Kagglehub cache.")

def load_split_df(dataset: str, split: str) -> pd.DataFrame:
    return pd.read_csv(find_dataset_file(DATASET_FILES[dataset][split]))

def load_multitask_data(dataset: str, split: str, limit: Optional[int] = None) -> List[MultiTaskSample]:
    df = load_split_df(dataset, split)
    if limit:
        df = df.head(limit)
    
    samples = []
    for idx, r in df.iterrows():
        if dataset == 'vifactcheck':
            sid = str(r.get('annotation_id') or idx)
            c = str(r.get('Statement') or '').strip()
            e = str(r.get('Evidence') or '').strip()
            raw_l = r.get('labels')
            l_map = {0: 'Supported', 1: 'Refuted', 2: 'NEI', '0': 'Supported', '1': 'Refuted', '2': 'NEI'}
        else:
            sid = str(r.get('pairID') or idx)
            c = str(r.get('claim') or '').strip()
            e = str(r.get('evidence') or '').strip()
            raw_l = r.get('gold_label')
            l_map = {'Supports': 'Supported', 'Refutes': 'Refuted', 'Not_Enough_Information': 'NEI'}
        
        l_str = l_map.get(raw_l)
        if not l_str or not c:
            continue
        
        c_type = detect_conflict_type(c, e, l_str)
        samples.append(MultiTaskSample(
            id=sid, claim=c, evidence=e,
            label_verdict=LABEL2ID[l_str],
            label_conflict=CONFLICT_MAP[c_type],
            conflict_name=c_type,
        ))
    
    print(f'Loaded {len(samples)} samples from {dataset} [{split}]')
    return samples

set_seed(ABLATION_SEED)
train_samples = load_multitask_data(DATASET_NAME, 'train', limit=TRAIN_LIMIT)
dev_samples = load_multitask_data(DATASET_NAME, 'dev', limit=DEV_LIMIT)
test_samples = load_multitask_data(DATASET_NAME, 'test', limit=TEST_LIMIT)


## 4. PyTorch Dataset & Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_KEY)
DIFF_OPEN, DIFF_CLOSE = '<diff>', '</diff>'
if USE_DIFF_MARKERS:
    tokenizer.add_special_tokens({'additional_special_tokens': [DIFF_OPEN, DIFF_CLOSE]})

_EDGE_PUNCT = re.compile(r'^\W+|\W+$')

def mark_claim_diff(claim: str, evidence: str) -> str:
    ev_words = {_EDGE_PUNCT.sub('', w.lower()) for w in evidence.split()}
    out, span = [], False
    for w in claim.split():
        norm = _EDGE_PUNCT.sub('', w.lower())
        novel = bool(norm) and norm not in ev_words
        if novel and not span:
            out.append(DIFF_OPEN); span = True
        elif not novel and span:
            out.append(DIFF_CLOSE); span = False
        out.append(w)
    if span:
        out.append(DIFF_CLOSE)
    return ' '.join(out)

def resize_with_mean_init(model, vocab_size: int):
    old_size = model.get_input_embeddings().weight.shape[0]
    if old_size != vocab_size:
        model.resize_token_embeddings(vocab_size)
        with torch.no_grad():
            emb = model.get_input_embeddings().weight
            emb[old_size:] = emb[:old_size].mean(dim=0, keepdim=True)

def encode_pair(first: str, second: str, max_len: int, use_diff: bool = True) -> Dict[str, torch.Tensor]:
    if use_diff and USE_DIFF_MARKERS:
        first = mark_claim_diff(first, second)
    enc = tokenizer(first, second, truncation=True, padding='max_length', max_length=max_len)
    seq = enc.sequence_ids()
    return {
        'input_ids': torch.tensor(enc['input_ids'], dtype=torch.long),
        'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
        'claim_mask': torch.tensor([1 if s == 0 else 0 for s in seq], dtype=torch.long),
        'evidence_mask': torch.tensor([1 if s == 1 else 0 for s in seq], dtype=torch.long),
    }

MODEL_INPUT_KEYS = ('input_ids', 'attention_mask', 'claim_mask', 'evidence_mask')
def model_inputs(batch) -> Dict[str, torch.Tensor]:
    return {k: batch[k].to(DEVICE) for k in MODEL_INPUT_KEYS}

class MultiTaskFactDataset(Dataset):
    def __init__(self, samples: List[MultiTaskSample], max_len: int = 256, use_diff: bool = True):
        self.samples = samples
        self.max_len = max_len
        self.use_diff = use_diff

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        item = encode_pair(s.claim, s.evidence, self.max_len, use_diff=self.use_diff)
        item['label_verdict'] = torch.tensor(s.label_verdict, dtype=torch.long)
        item['label_conflict'] = torch.tensor(s.label_conflict, dtype=torch.long)
        return item

train_loader = DataLoader(MultiTaskFactDataset(train_samples, MAX_LEN, use_diff=True), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(MultiTaskFactDataset(dev_samples, MAX_LEN, use_diff=True), batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(MultiTaskFactDataset(test_samples, MAX_LEN, use_diff=True), batch_size=BATCH_SIZE * 2, shuffle=False)

train_loader_nodiff = DataLoader(MultiTaskFactDataset(train_samples, MAX_LEN, use_diff=False), batch_size=BATCH_SIZE, shuffle=True)
dev_loader_nodiff = DataLoader(MultiTaskFactDataset(dev_samples, MAX_LEN, use_diff=False), batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader_nodiff = DataLoader(MultiTaskFactDataset(test_samples, MAX_LEN, use_diff=False), batch_size=BATCH_SIZE * 2, shuffle=False)

def get_loaders_for(cfg: Dict[str, Any]):
    if cfg.get('use_diff_markers', True):
        return train_loader, dev_loader, test_loader
    return train_loader_nodiff, dev_loader_nodiff, test_loader_nodiff

DEV_V_TRUES = np.array([s.label_verdict for s in dev_samples])
DEV_C_TRUES = np.array([s.label_conflict for s in dev_samples])
TEST_V_TRUES = np.array([s.label_verdict for s in test_samples])
TEST_C_TRUES = np.array([s.label_conflict for s in test_samples])
LABELS = ['Supported', 'Refuted', 'NEI']


## 5. Intermediate Vietnamese NLI Backbone

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

NLI_LABEL_TO_VERDICT = {'entailment': 0, 'contradiction': 1, 'neutral': 2}
XNLI_ID_TO_NAME = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
NLI_VAL_SAMPLES = 2000
NLI_FOLDER_NAME = f"xlmr_large_nli_{NLI_SOURCE}_{NLI_TRAIN_SAMPLES}_ep{NLI_EPOCHS}" + ('_diff' if USE_DIFF_MARKERS else '')
MAIN_RUNS_DIR = os.path.join(BASE_DIR, 'main_runs')

candidate_nli_dirs = [
    os.path.join(MAIN_RUNS_DIR, NLI_FOLDER_NAME),
    os.path.join(BASE_DIR, NLI_FOLDER_NAME),
    os.path.join(WORKING_DIR, NLI_FOLDER_NAME),
    f"/content/drive/MyDrive/NCKH/main_runs/{NLI_FOLDER_NAME}",
]

NLI_DIR = None
for c in candidate_nli_dirs:
    if c and os.path.exists(os.path.join(c, 'config.json')):
        NLI_DIR = c
        print(f"Found existing NLI backbone at: {NLI_DIR}")
        break

if NLI_DIR is None:
    NLI_DIR = os.path.join(MAIN_RUNS_DIR, NLI_FOLDER_NAME)

def load_nli_pairs():
    if NLI_SOURCE == 'xnli':
        from datasets import load_dataset
        train = load_dataset('facebook/xnli', 'vi', split='train').shuffle(seed=NLI_SEED).select(range(NLI_TRAIN_SAMPLES))
        val = load_dataset('facebook/xnli', 'vi', split='validation').shuffle(seed=NLI_SEED).select(range(NLI_VAL_SAMPLES))
        to_rows = lambda ds: [
            (r['hypothesis'], r['premise'], NLI_LABEL_TO_VERDICT[XNLI_ID_TO_NAME[int(r['label'])]]) for r in ds
        ]
        return to_rows(train), to_rows(val)
    raise ValueError(f"Unsupported NLI_SOURCE: {NLI_SOURCE}")

class NLIDataset(Dataset):
    def __init__(self, rows, max_len: int):
        self.rows = rows
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        h, p, label = self.rows[idx]
        item = encode_pair(h, p, self.max_len)
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

def nli_accuracy(model, loader) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch['input_ids'].to(DEVICE),
                           attention_mask=batch['attention_mask'].to(DEVICE)).logits
            correct += (logits.argmax(dim=-1).cpu() == batch['labels']).sum().item()
            total += batch['labels'].numel()
    return correct / max(total, 1)

def run_nli_intermediate():
    train_rows, val_rows = load_nli_pairs()
    print(f'NLI {NLI_SOURCE}: train {len(train_rows)} | val {len(val_rows)} -> {NLI_DIR}')
    set_seed(NLI_SEED)
    nli_model = AutoModelForSequenceClassification.from_pretrained(MODEL_KEY, num_labels=3)
    resize_with_mean_init(nli_model, len(tokenizer))
    nli_model.to(DEVICE)

    loader = DataLoader(NLIDataset(train_rows, NLI_MAX_LEN), batch_size=NLI_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(NLIDataset(val_rows, NLI_MAX_LEN), batch_size=NLI_BATCH_SIZE * 2, shuffle=False)
    optimizer = torch.optim.AdamW(nli_model.parameters(), lr=NLI_LR, weight_decay=0.01, eps=1e-8)
    total_steps = max(1, len(loader) * NLI_EPOCHS // NLI_GRAD_ACCUM)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.06), total_steps)
    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0.0
    for epoch in range(NLI_EPOCHS):
        nli_model.train()
        t0, running = time.time(), 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(loader):
            with torch.cuda.amp.autocast():
                out = nli_model(input_ids=batch['input_ids'].to(DEVICE),
                                attention_mask=batch['attention_mask'].to(DEVICE),
                                labels=batch['labels'].to(DEVICE))
                loss = out.loss / NLI_GRAD_ACCUM
            scaler.scale(loss).backward()
            running += out.loss.item()
            if (step + 1) % NLI_GRAD_ACCUM == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(nli_model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()

        val_acc = nli_accuracy(nli_model, val_loader)
        print(f'Epoch {epoch+1}/{NLI_EPOCHS} | loss {running/len(loader):.4f} | val acc {val_acc*100:.2f}% | {time.time()-t0:.0f}s')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs(NLI_DIR, exist_ok=True)
            nli_model.base_model.save_pretrained(NLI_DIR)
            tokenizer.save_pretrained(NLI_DIR)

    del nli_model, optimizer
    import gc
    gc.collect()
    torch.cuda.empty_cache()

def backbone_for(cfg) -> str:
    return NLI_DIR if cfg['use_nli'] else MODEL_KEY

if any(cfg['use_nli'] for cfg in ACTIVE_ABLATIONS):
    if not os.path.exists(os.path.join(NLI_DIR, 'config.json')) and not FORCE_RETRAIN_NLI:
        print(f'Training NLI intermediate ({NLI_EPOCHS} epochs) -> {NLI_DIR}')
        run_nli_intermediate()
    else:
        print(f'Using NLI backbone at: {NLI_DIR}')


## 6. SAMV-FC Architecture

In [ ]:
from transformers import AutoModel

class SAMVFactClassifier(nn.Module):
    def __init__(self, model_name: str, num_verdict_labels: int = 3, num_conflict_labels: int = 5,
                 dropout: float = 0.1, pair_pooling: bool = True,
                 use_conflict_head: bool = True, use_gate: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.pair_pooling = pair_pooling
        self.use_conflict_head = use_conflict_head
        self.use_gate = use_gate and use_conflict_head

        if pair_pooling:
            self.pair_proj = nn.Sequential(nn.Linear(5 * hidden_size, hidden_size), nn.GELU(), nn.Dropout(dropout))

        if use_conflict_head:
            self.conflict_proj = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(dropout))
            self.conflict_head = nn.Linear(hidden_size // 2, num_conflict_labels)
        if self.use_gate:
            self.conflict_gate = nn.Sequential(nn.Linear(hidden_size // 2, hidden_size), nn.Sigmoid())
            self.conflict_shift_proj = nn.Linear(hidden_size // 2, hidden_size)

        self.verdict_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_verdict_labels)
        )

    @staticmethod
    def _masked_mean(hidden, mask):
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1.0)

    def forward(self, input_ids, attention_mask, claim_mask=None, evidence_mask=None):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        cls_rep = hidden[:, 0, :]
        if self.pair_pooling:
            claim_rep = self._masked_mean(hidden, claim_mask)
            evidence_rep = self._masked_mean(hidden, evidence_mask)
            pair = torch.cat([cls_rep, claim_rep, evidence_rep,
                              (claim_rep - evidence_rep).abs(), claim_rep * evidence_rep], dim=-1)
            rep = self.pair_proj(self.dropout(pair))
        else:
            rep = self.dropout(cls_rep)

        logits_conflict = None
        if self.use_conflict_head:
            conflict_feat = self.conflict_proj(rep)
            logits_conflict = self.conflict_head(conflict_feat)
            if self.use_gate:
                rep = rep + self.conflict_gate(conflict_feat) * self.conflict_shift_proj(conflict_feat)

        return self.verdict_head(rep), logits_conflict

ConflictGatedFactClassifier = SAMVFactClassifier

def build_model(cfg) -> SAMVFactClassifier:
    model = SAMVFactClassifier(
        backbone_for(cfg), dropout=DROPOUT, pair_pooling=cfg.get('use_pair_pooling', USE_PAIR_POOLING),
        use_conflict_head=cfg['use_conflict_head'], use_gate=cfg['use_gate'],
    )
    resize_with_mean_init(model.backbone, len(tokenizer))
    return model.to(DEVICE)


## 7. Ablation Training Loop

In [ ]:
class_weights = torch.tensor([1.0, 1.20, 1.0], device=DEVICE)
loss_fn_verdict = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
loss_fn_conflict = nn.CrossEntropyLoss()
RESULTS_JSON_PATH = os.path.join(OUT_DIR, 'ablation_results.json')

def mcnemar_p(correct_a, correct_b) -> float:
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    if b + c == 0:
        return 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return math.erfc(math.sqrt(chi2 / 2))

def compute_eval_metrics(v_probs, c_probs, v_trues, c_trues):
    pred = v_probs.argmax(axis=1)
    report = classification_report(v_trues, pred, target_names=LABELS, output_dict=True, digits=4, zero_division=0)
    slot_acc = None
    if c_probs is not None and c_probs.size:
        idx = np.where(v_trues == 1)[0]
        if len(idx) > 0:
            slot_acc = float(accuracy_score(c_trues[idx], c_probs.argmax(axis=1)[idx]))
    return {
        'macro_f1': float(f1_score(v_trues, pred, average='macro', zero_division=0)),
        'accuracy': float(accuracy_score(v_trues, pred)),
        'per_class_f1': {lab: float(report[lab]['f1-score']) for lab in LABELS},
        'conflict_slot_acc': slot_acc,
        'confusion_matrix': confusion_matrix(v_trues, pred).tolist(),
        '_correct': (pred == v_trues),
        '_pred': pred,
    }

def ablation_probs_path(name: str) -> str:
    return os.path.join(OUT_DIR, f'{name}_probs.npz')

def is_ablation_completed(name: str) -> bool:
    probs_path = ablation_probs_path(name)
    if not os.path.exists(probs_path) or os.path.getsize(probs_path) == 0:
        return False
    try:
        with np.load(probs_path) as d:
            return 'dev_verdict' in d and 'test_verdict' in d
    except Exception:
        return False

def import_full_result_if_available() -> bool:
    full_probs = ablation_probs_path('full')
    if is_ablation_completed('full'):
        return True
    if FULL_REFERENCE_NPZ and os.path.exists(FULL_REFERENCE_NPZ):
        try:
            d = np.load(FULL_REFERENCE_NPZ)
            if 'test_verdict' in d and len(d['test_verdict']) == len(test_samples):
                np.savez(
                    full_probs,
                    dev_verdict=d['dev_verdict'],
                    dev_conflict=d.get('dev_conflict', np.zeros((0, 0))),
                    test_verdict=d['test_verdict'],
                    test_conflict=d.get('test_conflict', np.zeros((0, 0))),
                    best_epoch=int(d.get('best_epoch', 1)),
                    train_minutes=0.0,
                    config=json.dumps(FULL, ensure_ascii=False)
                )
                print(f"Imported full reference predictions from: {FULL_REFERENCE_NPZ}")
                return True
        except Exception as e:
            print(f"Warning reading FULL_REFERENCE_NPZ: {e}")
    return False

def load_full_reference():
    label = f'SAMV-FC Large (Full, seed {ABLATION_SEED})'
    full_probs = ablation_probs_path('full')
    if os.path.exists(full_probs):
        d = np.load(full_probs)
        dev_m = compute_eval_metrics(d['dev_verdict'], d['dev_conflict'], DEV_V_TRUES, DEV_C_TRUES)
        test_m = compute_eval_metrics(d['test_verdict'], d['test_conflict'], TEST_V_TRUES, TEST_C_TRUES)
        return {
            'label': label,
            'config': FULL,
            'best_epoch': int(d['best_epoch']),
            'train_minutes': round(float(d['train_minutes']), 1),
            'dev_macro_f1': dev_m['macro_f1'],
            'dev_accuracy': dev_m['accuracy'],
            'test_macro_f1': test_m['macro_f1'],
            'test_accuracy': test_m['accuracy'],
            'per_class_f1': test_m['per_class_f1'],
            'conflict_slot_acc': test_m['conflict_slot_acc'],
            'confusion_matrix': test_m['confusion_matrix'],
            '_correct_test': test_m['_correct'],
            '_pred_test': test_m['_pred'],
        }
    m = FULL_REFERENCE_METRICS
    return {
        'label': label + ' [Reference]',
        'config': FULL,
        'best_epoch': None,
        'train_minutes': None,
        'dev_macro_f1': m['dev_macro_f1'],
        'dev_accuracy': None,
        'test_macro_f1': m['test_macro_f1'],
        'test_accuracy': m['test_accuracy'],
        'per_class_f1': m['per_class_f1'],
        'conflict_slot_acc': m.get('conflict_slot_acc'),
        'confusion_matrix': None,
        '_correct_test': None,
        '_pred_test': None,
    }

def generate_and_print_ablation_table():
    current_results = {}
    for cfg in ABLATIONS:
        name = cfg['name']
        if not is_ablation_completed(name):
            continue
        try:
            d = np.load(ablation_probs_path(name))
            dev_m = compute_eval_metrics(d['dev_verdict'], d['dev_conflict'], DEV_V_TRUES, DEV_C_TRUES)
            test_m = compute_eval_metrics(d['test_verdict'], d['test_conflict'], TEST_V_TRUES, TEST_C_TRUES)
            current_results[name] = {
                'label': cfg['label'],
                'config': json.loads(str(d['config'])),
                'best_epoch': int(d['best_epoch']),
                'train_minutes': round(float(d['train_minutes']), 1),
                'dev_macro_f1': dev_m['macro_f1'],
                'dev_accuracy': dev_m['accuracy'],
                'test_macro_f1': test_m['macro_f1'],
                'test_accuracy': test_m['accuracy'],
                'per_class_f1': test_m['per_class_f1'],
                'conflict_slot_acc': test_m['conflict_slot_acc'],
                'confusion_matrix': test_m['confusion_matrix'],
                '_correct_test': test_m['_correct'],
                '_pred_test': test_m['_pred'],
            }
        except Exception as e:
            print(f"Warning reading {name}: {e}")
    
    ordered_results = {}
    ordered_results['full'] = current_results['full'] if 'full' in current_results else load_full_reference()
    full_row = ordered_results['full']
    
    for cfg in ABLATIONS:
        n = cfg['name']
        if n != 'full' and n in current_results:
            ordered_results[n] = current_results[n]
    current_results = ordered_results
    
    pct = lambda x: '—' if x is None else f'{x*100:.2f}'
    header = ('| Configuration | Dev F1 | Test F1 | Test Acc | Supported F1 | Refuted F1 | NEI F1 | '
              'Conflict slot acc | ΔF1 vs Full | McNemar p vs Full |')
    lines = [header, '|' + '---|' * 10]
    
    for name, r in current_results.items():
        delta = p_val = '—'
        if name != 'full':
            delta = f"{(r['test_macro_f1'] - full_row['test_macro_f1'])*100:+.2f}"
            if full_row.get('_correct_test') is not None and r.get('_correct_test') is not None:
                p_val = f"{mcnemar_p(full_row['_correct_test'], r['_correct_test']):.3f}"
        lines.append(
            f"| {r['label']} | {pct(r.get('dev_macro_f1'))} | {pct(r['test_macro_f1'])} | {pct(r['test_accuracy'])} | "
            f"{pct(r['per_class_f1']['Supported'])} | {pct(r['per_class_f1']['Refuted'])} | {pct(r['per_class_f1']['NEI'])} | "
            f"{pct(r['conflict_slot_acc'])} | {delta} | {p_val} |"
        )
    table_md = '\n'.join(lines)
    
    print('\n' + '=' * 105)
    print(f'ABLATION RESULTS ON TEST SET ({DATASET_NAME.upper()} | seed {ABLATION_SEED} | test n={len(test_samples)}):')
    print('=' * 105)
    print(table_md)
    print('=' * 105 + '\n')
    
    with open(os.path.join(OUT_DIR, 'ablation_table.md'), 'w', encoding='utf-8') as f:
        f.write(table_md + '\n')
    
    pd.DataFrame([
        {'name': n, 'label': r['label'], 'dev_macro_f1': r.get('dev_macro_f1'),
         'test_macro_f1': r['test_macro_f1'], 'test_accuracy': r['test_accuracy'],
         **{f'test_f1_{k}': v for k, v in r['per_class_f1'].items()},
         'test_conflict_slot_acc': r['conflict_slot_acc'],
         'best_epoch': r['best_epoch'], 'train_minutes': r['train_minutes']}
        for n, r in current_results.items()
    ]).to_csv(os.path.join(OUT_DIR, 'ablation_results.csv'), index=False)
    
    with open(RESULTS_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump({
            'dataset': DATASET_NAME,
            'seed': ABLATION_SEED,
            'eval_split': 'test',
            'num_test_samples': len(test_samples),
            'num_dev_samples': len(dev_samples),
            'results': {n: {k: v for k, v in r.items() if not k.startswith('_')} for n, r in current_results.items()},
            'saved_at': datetime.now(timezone.utc).isoformat(),
        }, f, indent=2, ensure_ascii=False)
    
    return current_results

def record_ablation_predictions(name: str, test_v):
    pred = test_v.argmax(axis=1)
    pred_path = os.path.join(OUT_DIR, f'test_predictions_{name}.json')
    with open(pred_path, 'w', encoding='utf-8') as f:
        json.dump([
            {'id': s.id, 'gold_verdict': ID2LABEL[int(g)], 'pred_verdict': ID2LABEL[int(p)]}
            for s, g, p in zip(test_samples, TEST_V_TRUES, pred)
        ], f, ensure_ascii=False)

def predict_probs(model, loader):
    model.eval()
    v_probs, c_probs = [], []
    with torch.no_grad():
        for batch in loader:
            logits_v, logits_c = model(**model_inputs(batch))
            v_probs.append(torch.softmax(logits_v.float(), dim=-1).cpu().numpy())
            if logits_c is not None:
                c_probs.append(torch.softmax(logits_c.float(), dim=-1).cpu().numpy())
    return np.concatenate(v_probs), (np.concatenate(c_probs) if c_probs else None)

def train_one(cfg, seed: int, ckpt_path: str):
    set_seed(seed)
    model = build_model(cfg)
    lam_c, lam_cons = cfg['lambda_conflict'], cfg['lambda_consistency']
    cur_train_loader, cur_dev_loader, cur_test_loader = get_loaders_for(cfg)
    
    ref_weight = cfg.get('refuted_weight', 1.20)
    cur_weights = torch.tensor([1.0, ref_weight, 1.0], device=DEVICE)
    cur_smoothing = cfg.get('label_smoothing', LABEL_SMOOTHING)
    cur_loss_fn_verdict = nn.CrossEntropyLoss(weight=cur_weights, label_smoothing=cur_smoothing)
    
    backbone_params = list(model.backbone.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE, 'weight_decay': 1e-2},
        {'params': head_params, 'lr': LR_HEADS, 'weight_decay': 1e-3}
    ], eps=1e-8)
    total_steps = (len(cur_train_loader) // GRAD_ACCUM) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
    scaler = torch.cuda.amp.GradScaler()
    
    best_f1, best_epoch = -1.0, 0
    print(f"\n--- [{cfg['name']}] Seed {seed} | {cfg['label']} ---")
    for epoch in range(NUM_EPOCHS):
        backbone_trainable = epoch >= HEAD_WARMUP_EPOCHS
        for p in model.backbone.parameters():
            p.requires_grad = backbone_trainable
        
        model.train()
        sums = {'v': 0.0, 'c': 0.0, 'cons': 0.0, 'kl': 0.0}
        t0 = time.time()
        optimizer.zero_grad()
        for step, batch in enumerate(cur_train_loader):
            inputs = model_inputs(batch)
            labels_v = batch['label_verdict'].to(DEVICE)
            labels_c = batch['label_conflict'].to(DEVICE)
            
            with torch.cuda.amp.autocast():
                if R_DROP_ALPHA > 0:
                    doubled = {k: torch.cat([v, v], dim=0) for k, v in inputs.items()}
                    labels_v = torch.cat([labels_v, labels_v], dim=0)
                    labels_c = torch.cat([labels_c, labels_c], dim=0)
                    logits_v, logits_c = model(**doubled)
                    logp_a, logp_b = F.log_softmax(logits_v.float(), dim=-1).chunk(2, dim=0)
                    l_kl = 0.5 * (F.kl_div(logp_a, logp_b, reduction='batchmean', log_target=True)
                                  + F.kl_div(logp_b, logp_a, reduction='batchmean', log_target=True))
                else:
                    logits_v, logits_c = model(**inputs)
                    l_kl = torch.zeros((), device=DEVICE)
                
                zero = torch.zeros((), device=DEVICE)
                l_v = cur_loss_fn_verdict(logits_v, labels_v)
                l_c = loss_fn_conflict(logits_c, labels_c) if (logits_c is not None and lam_c > 0) else zero
                if logits_c is not None and lam_cons > 0:
                    prob_v = torch.softmax(logits_v, dim=-1)
                    prob_c = torch.softmax(logits_c, dim=-1)
                    l_consist = torch.mean(prob_v[:, 0] * (1.0 - prob_c[:, 0]))
                else:
                    l_consist = zero
                loss = (l_v + lam_c * l_c + lam_cons * l_consist + R_DROP_ALPHA * l_kl) / GRAD_ACCUM
            
            scaler.scale(loss).backward()
            sums['v'] += l_v.item(); sums['c'] += l_c.item(); sums['cons'] += l_consist.item(); sums['kl'] += l_kl.item()
            
            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(cur_train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()
        
        dev_v_probs, _ = predict_probs(model, cur_dev_loader)
        dev_macro_f1 = f1_score(DEV_V_TRUES, dev_v_probs.argmax(axis=1), average='macro', zero_division=0)
        n = len(cur_train_loader)
        phase_str = 'FULL' if backbone_trainable else 'WARMUP'
        print(f"[{cfg['name']}] Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} [{time.time()-t0:.0f}s] ({phase_str}) | "
              f"Loss(V): {sums['v']/n:.3f} | Dev Macro-F1: {dev_macro_f1*100:.2f}%")
        if dev_macro_f1 > best_f1:
            best_f1, best_epoch = dev_macro_f1, epoch + 1
            torch.save(model.state_dict(), ckpt_path)
            print(f'  --> [BEST DEV] Dev F1: {dev_macro_f1*100:.2f}% (Epoch {best_epoch})')
    return model, best_epoch

has_full = import_full_result_if_available()
print(f"Starting ablation study ({len(ACTIVE_ABLATIONS)} configurations selected)...")

for cfg in ACTIVE_ABLATIONS:
    name = cfg['name']
    probs_path = ablation_probs_path(name)
    if name == 'full' and is_ablation_completed('full'):
        print(f"Skipping 'full' - already imported from main_runs.")
        generate_and_print_ablation_table()
        continue
    if is_ablation_completed(name):
        print(f"Skipping '{name}' - cached results exist at {probs_path}.")
        generate_and_print_ablation_table()
        continue
    
    ckpt_path = os.path.join(OUT_DIR, f"{name}_seed{ABLATION_SEED}.pt")
    t_start = time.time()
    model, best_epoch = train_one(cfg, ABLATION_SEED, ckpt_path)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    
    cur_train_loader, cur_dev_loader, cur_test_loader = get_loaders_for(cfg)
    dev_v, dev_c = predict_probs(model, cur_dev_loader)
    test_v, test_c = predict_probs(model, cur_test_loader)
    train_mins = (time.time() - t_start) / 60
    
    np.savez(
        probs_path,
        dev_verdict=dev_v,
        dev_conflict=dev_c if dev_c is not None else np.zeros((0, 0)),
        test_verdict=test_v,
        test_conflict=test_c if test_c is not None else np.zeros((0, 0)),
        best_epoch=best_epoch,
        train_minutes=train_mins,
        config=json.dumps(cfg, ensure_ascii=False),
    )
    record_ablation_predictions(name, test_v)
    print(f"Completed [{name}] in {train_mins:.1f}m | Best Epoch: {best_epoch} | Saved: {probs_path}")
    generate_and_print_ablation_table()
    
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    if not KEEP_CHECKPOINTS and os.path.exists(ckpt_path):
        os.remove(ckpt_path)

print('\nAblation experiments completed!')


## 8. Ablation Evaluation & Visualization

In [ ]:
results = generate_and_print_ablation_table()
print(f"Markdown Table : {os.path.join(OUT_DIR, 'ablation_table.md')}")
print(f"CSV Results    : {os.path.join(OUT_DIR, 'ablation_results.csv')}")
print(f"JSON Results   : {RESULTS_JSON_PATH}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5), dpi=150)
names = [r['label'] for r in results.values()]
f1_scores = [r.get('test_macro_f1', r.get('dev_macro_f1', 0)) * 100 for r in results.values()]
colors = ['#2b5c8f' if 'Full' in n else '#e76f51' for n in names]
bars = plt.barh(names, f1_scores, color=colors, height=0.55)

for bar in bars:
    width = bar.get_width()
    if width > 0:
        plt.text(width + 0.3, bar.get_y() + bar.get_height()/2, f"{width:.2f}%",
                 va='center', ha='left', fontsize=10, fontweight='bold')

plt.xlim(min(f1_scores) - 3 if min(f1_scores) > 3 else 0, max(f1_scores) + 3)
plt.xlabel('Test Macro-F1 (%)', fontsize=11, fontweight='bold')
plt.title(f'Ablation Study Comparison on Test Set ({DATASET_NAME.upper()})', fontsize=12, fontweight='bold', pad=12)
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

chart_path = os.path.join(OUT_DIR, 'ablation_test_f1_comparison.png')
plt.savefig(chart_path, bbox_inches='tight')
plt.show()
print(f'Saved ablation comparison plot to: {chart_path}')
